## 03a: Wearable Data Pulls

Selects valid days and valid nights of Fitbit data for MDD, BD and HC participants from the
wearable cache built by `00_build_cache`.

- **Valid day:** at least 10 hours each with ≥20 minutes of heart rate recording, and
  250–45,000 steps.
- **Valid night:** sleep onset between 18:00 and 08:00, 120–720 minutes of total sleep, and no
  more than 720 minutes from onset to offset.

Data are read once per participant over a window covering all their tasks, from 90 days before
the first task to the last task date; `03b` then cuts each task's own 90-day window.

Outputs: `raw_pulls/valid_days.parquet` and `raw_pulls/valid_nights.parquet`.

## Task 0: Setup

In [ ]:
import sys
import os
import gc
import pandas as pd

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils import *
from src.wearables_cache import iter_wearables_shards

OVERWRITE = False
# False: reuse valid_days / valid_nights from the bucket if present

# -- Validity thresholds -------------------------------------------------------
N_DAYS_WEARABLE   = 90        # calendar length of per-task feature window
STEP_MIN          = 250       # daily steps lower bound
STEP_MAX          = 45_000    # daily steps upper bound
MIN_HR_MINUTES    = 20        # min valid HR minutes per hour
HR_HOURS_MIN      = 10        # min valid HR hours per day
SLEEP_DUR_MIN     = 120       # min minute_asleep for a valid night
SLEEP_DUR_MAX     = 720       # max minute_asleep for a valid night
ONSET_BLOCK_START = 8         # onset hours in [8, 18) mark an invalid night
ONSET_BLOCK_END   = 18

# -- Eligibility gate (applied in 03b; previewed in Task 3) --------------------
N_REC_DAYS_MIN   = 30         # min valid days  within task window
N_REC_NIGHTS_MIN = 30         # min valid nights within task window

# -- Output paths -------------------------------------------------------------
VDAYS_OUT = 'raw_pulls/valid_days.parquet'
VN_OUT    = 'raw_pulls/valid_nights.parquet'

# -- Load cohort and include all groups (MDD, BD, HC) -------------------------
cohort          = load_parquet('cohort/cohort_master.parquet')
cohort_wearable = cohort[cohort['group'].isin(['MDD', 'BD', 'HC'])].copy()

# Parse date columns
ETM_TASKS = ['gradcpt', 'delaydiscounting', 'flanker', 'emorecog']
date_cols = [f'etm_date_{t}' for t in ETM_TASKS]
for col in date_cols + ['etm_date_first']:
    cohort_wearable[col] = pd.to_datetime(cohort_wearable[col])

# -- Union window per participant ---------------------------------------------
# t0_union    = etm_date_first − N_DAYS_WEARABLE  (= min task date − 90 days)
# t_end_union = max(etm_date_{completed tasks})
cohort_wearable['t0_union']    = cohort_wearable['etm_date_first'] - pd.Timedelta(days=N_DAYS_WEARABLE)
cohort_wearable['t_end_union'] = cohort_wearable[date_cols].max(axis=1)

pids         = sorted(cohort_wearable['person_id'].tolist())
t0_map       = dict(zip(cohort_wearable['person_id'], cohort_wearable['t0_union']))
t_end_map    = dict(zip(cohort_wearable['person_id'], cohort_wearable['t_end_union']))
etm_date_map = dict(zip(cohort_wearable['person_id'], cohort_wearable['etm_date_first']))

span_days = (cohort_wearable['t_end_union'] - cohort_wearable['t0_union']).dt.days
n_mdd = (cohort_wearable['group'] == 'MDD').sum()
n_bd  = (cohort_wearable['group'] == 'BD').sum()
n_hc  = (cohort_wearable['group'] == 'HC').sum()
print(f"MDD + BD + HC participants  : {len(pids):,}  (MDD: {n_mdd:,}, BD: {n_bd:,}, HC: {n_hc:,})")
print(f"Union window range          : {cohort_wearable['t0_union'].min().date()} → {cohort_wearable['t_end_union'].max().date()}")
print(f"Window width (days)         : median={span_days.median():.0f}  mean={span_days.mean():.1f}  max={span_days.max()}")
print(f"OVERWRITE                   : {OVERWRITE}")


## Task 1: Valid Days

Daily steps and the number of hours with ≥20 minutes of heart rate recording, from the hourly
cache; days meeting both criteria are kept.

In [ ]:
if not gcs_exists(VDAYS_OUT) or OVERWRITE:

    # -- 1a: Daily steps from steps_hourly ------------------------------------
    print('=== 1a: Streaming steps_hourly -> daily steps ===')
    step_chunks = []
    for shard_df in iter_wearables_shards(pids, 'steps_hourly'):
        daily = (
            shard_df
            .groupby(['person_id', 'date'], as_index=False)['hourly_steps']
            .sum()
            .rename(columns={'hourly_steps': 'steps'})
        )
        daily['date']   = pd.to_datetime(daily['date'])
        daily['_t0']    = daily['person_id'].map(t0_map)
        daily['_t_end'] = daily['person_id'].map(t_end_map)
        daily = daily[
            daily['_t0'].notna()
            & (daily['date'] >= daily['_t0'])
            & (daily['date'] <= daily['_t_end'])
        ].drop(columns=['_t0', '_t_end'])
        step_chunks.append(daily)
        del shard_df, daily
        gc.collect()

    steps_daily = pd.concat(step_chunks, ignore_index=True)
    del step_chunks
    gc.collect()
    print(f"steps_daily: {len(steps_daily):,} rows for {steps_daily['person_id'].nunique():,} pids\n")

    # -- 1b: Valid HR hours from hr_hourly ------------------------------------
    print('=== 1b: Streaming hr_hourly -> daily valid HR hours ===')
    hr_chunks = []
    for shard_df in iter_wearables_shards(pids, 'hr_hourly'):
        shard_df['date'] = pd.to_datetime(shard_df['date'])
        shard_df['_t0']    = shard_df['person_id'].map(t0_map)
        shard_df['_t_end'] = shard_df['person_id'].map(t_end_map)
        shard_df = shard_df[
            shard_df['_t0'].notna()
            & (shard_df['date'] >= shard_df['_t0'])
            & (shard_df['date'] <= shard_df['_t_end'])
        ].drop(columns=['_t0', '_t_end'])
        valid_hrs = (
            shard_df[shard_df['n_valid_minutes'] >= MIN_HR_MINUTES]
            .groupby(['person_id', 'date'])
            .size()
            .reset_index(name='valid_hr_hours')
        )
        hr_chunks.append(valid_hrs)
        del shard_df, valid_hrs
        gc.collect()

    hr_daily = pd.concat(hr_chunks, ignore_index=True)
    del hr_chunks
    gc.collect()
    print(f"hr_daily: {len(hr_daily):,} rows for {hr_daily['person_id'].nunique():,} pids\n")

    # -- 1c: Merge, apply validity mask, add day_delta ------------------------
    days = steps_daily.merge(hr_daily, on=['person_id', 'date'], how='left')
    days['valid_hr_hours'] = days['valid_hr_hours'].fillna(0).astype(int)
    del steps_daily, hr_daily
    gc.collect()

    n_before = len(days)
    valid_mask = (
        (days['steps'] > STEP_MIN)
        & (days['steps'] < STEP_MAX)
        & (days['valid_hr_hours'] >= HR_HOURS_MIN)
    )
    valid_days_df = days[valid_mask].copy()
    print(f"Validity filter: {n_before:,} candidate days -> {len(valid_days_df):,} valid days "
          f"({100*len(valid_days_df)/n_before:.1f}% pass rate)")
    del days
    gc.collect()

    valid_days_df['day_delta'] = (
        valid_days_df['date'] - valid_days_df['person_id'].map(etm_date_map)
    ).dt.days.astype(int)

    save_parquet(valid_days_df, VDAYS_OUT)
    print(f"\nvalid_days: {len(valid_days_df):,} rows for {valid_days_df['person_id'].nunique():,} participants (MDD, BD, HC)")
    print(f"day_delta range: [{valid_days_df['day_delta'].min()}, {valid_days_df['day_delta'].max()}]")
    del valid_days_df
    gc.collect()

else:
    print(f'Checkpoint found -- skipping: {VDAYS_OUT}')

## Task 2: Valid Nights

Main sleep episodes (onset/offset times and the nightly sleep summary) meeting the three night
criteria. `night_date` is the evening the sleep began; `sleep_date` is the morning after.

In [ ]:
if not gcs_exists(VN_OUT) or OVERWRITE:

    # -- 2a: Stream sleep_onset_offset, filter to [t0_union, t_end_union] on night_date --
    print('=== 2a: Streaming sleep_onset_offset -> window-filtered SOO ===')
    soo_chunks = []
    for shard_df in iter_wearables_shards(pids, 'sleep_onset_offset'):
        for col in ['sleep_date', 'night_date', 'onset_dt', 'offset_dt']:
            shard_df[col] = pd.to_datetime(shard_df[col])
        shard_df['_t0']    = shard_df['person_id'].map(t0_map)
        shard_df['_t_end'] = shard_df['person_id'].map(t_end_map)
        shard_df = shard_df[
            shard_df['_t0'].notna()
            & (shard_df['night_date'] >= shard_df['_t0'])
            & (shard_df['night_date'] <= shard_df['_t_end'])
        ].drop(columns=['_t0', '_t_end'])
        if len(shard_df) > 0:
            soo_chunks.append(shard_df)
        del shard_df
        gc.collect()

    soo_filtered = pd.concat(soo_chunks, ignore_index=True)
    del soo_chunks
    gc.collect()
    print(f"SOO after window filter: {len(soo_filtered):,} rows for {soo_filtered['person_id'].nunique():,} pids\n")

    # -- 2b: Stream sleep_summary, filter, join with SOO, apply validity ------
    print('=== 2b: Streaming sleep_summary -> merge with SOO -> apply validity ===')
    result_chunks      = []
    n_dropped_no_soo   = 0
    n_dropped_validity = 0

    for shard_df in iter_wearables_shards(pids, 'sleep_summary'):
        shard_df['sleep_date'] = pd.to_datetime(shard_df['sleep_date'])
        shard_df['_t0']    = shard_df['person_id'].map(t0_map)
        shard_df['_t_end'] = shard_df['person_id'].map(t_end_map)
        # sleep_date = wakeup date = night_date + 1 day; shift window by +1 day
        shard_df = shard_df[
            shard_df['_t0'].notna()
            & (shard_df['sleep_date'] >= shard_df['_t0'] + pd.Timedelta(days=1))
            & (shard_df['sleep_date'] <= shard_df['_t_end'] + pd.Timedelta(days=1))
        ].drop(columns=['_t0', '_t_end'])

        if len(shard_df) == 0:
            del shard_df; gc.collect(); continue

        shard_pids = shard_df['person_id'].unique()
        soo_shard  = soo_filtered[soo_filtered['person_id'].isin(shard_pids)]

        n_before = len(shard_df)
        merged = shard_df.merge(soo_shard, on=['person_id', 'sleep_date'], how='inner')
        n_dropped_no_soo += n_before - len(merged)

        if len(merged) == 0:
            del shard_df, soo_shard, merged; gc.collect(); continue

        # -- Validity criteria ------------------------------------------------
        onset_hour = merged['onset_dt'].dt.hour

        # 1. minute_asleep in [SLEEP_DUR_MIN, SLEEP_DUR_MAX]
        valid_duration = (
            (merged['minute_asleep'] >= SLEEP_DUR_MIN)
            & (merged['minute_asleep'] <= SLEEP_DUR_MAX)
        )
        # 2. Onset NOT in [ONSET_BLOCK_START, ONSET_BLOCK_END)
        valid_onset = ~(
            (onset_hour >= ONSET_BLOCK_START)
            & (onset_hour < ONSET_BLOCK_END)
        )
        # 3. Onset to offset <= SLEEP_DUR_MAX (keeps offsets out of the afternoon)
        episode_minutes = (
            (merged['offset_dt'] - merged['onset_dt']).dt.total_seconds() / 60
        )
        valid_episode_dur = episode_minutes <= SLEEP_DUR_MAX

        n_before_validity = len(merged)
        valid_mask = valid_duration & valid_onset & valid_episode_dur
        result_chunks.append(merged[valid_mask].copy())
        n_dropped_validity += n_before_validity - valid_mask.sum()

        del shard_df, soo_shard, merged
        gc.collect()

    del soo_filtered
    gc.collect()

    print(f"\nDropped {n_dropped_no_soo:,} sleep_summary rows with no matching SOO record")
    print(f"Dropped {n_dropped_validity:,} rows failing validity criteria "
          f"(minute_asleep range, onset block, or episode duration > {SLEEP_DUR_MAX} min)")

    valid_nights_df = pd.concat(result_chunks, ignore_index=True)
    del result_chunks
    gc.collect()

    # -- 2c: Add day_delta, select and order columns --------------------------
    valid_nights_df['day_delta'] = (
        valid_nights_df['night_date'] - valid_nights_df['person_id'].map(etm_date_map)
    ).dt.days.astype(int)

    keep_cols = [
        'person_id', 'night_date', 'sleep_date',
        'minute_asleep', 'minute_awake', 'minute_in_bed', 'minute_wake',
        'onset_dt', 'offset_dt', 'day_delta',
    ]
    # minute_wake is NaN for classic (non-stages) devices; retain for completeness
    missing = [c for c in keep_cols if c not in valid_nights_df.columns]
    if missing:
        print(f'WARNING: expected columns absent from merge result: {missing}')
    valid_nights_df = valid_nights_df[[c for c in keep_cols if c in valid_nights_df.columns]]

    save_parquet(valid_nights_df, VN_OUT)
    print(f"\nvalid_nights: {len(valid_nights_df):,} rows for {valid_nights_df['person_id'].nunique():,} participants (MDD, BD, HC)")
    print(f"day_delta range: [{valid_nights_df['day_delta'].min()}, {valid_nights_df['day_delta'].max()}]")
    del valid_nights_df
    gc.collect()

else:
    print(f'Checkpoint found -- skipping: {VN_OUT}')

## Task 3: QC Summary

Valid days and nights per participant over the whole window. The per-task eligibility
(≥30 valid days and ≥30 valid nights) is applied in `03b`.

In [ ]:
vd = load_parquet(VDAYS_OUT)
vn = load_parquet(VN_OUT)
vd['date']       = pd.to_datetime(vd['date'])
vn['night_date'] = pd.to_datetime(vn['night_date'])

# Per-participant valid counts over the union window
vd_counts = vd.groupby('person_id').size().rename('n_valid_days').reset_index()
vn_counts = vn.groupby('person_id').size().rename('n_valid_nights').reset_index()

summary = (
    cohort_wearable[['person_id', 'group']]
    .merge(vd_counts, on='person_id', how='left')
    .merge(vn_counts, on='person_id', how='left')
)
summary['n_valid_days']   = summary['n_valid_days'].fillna(0).astype(int)
summary['n_valid_nights'] = summary['n_valid_nights'].fillna(0).astype(int)

n = len(summary)
n_day_elig = (summary['n_valid_days']   >= N_REC_DAYS_MIN).sum()
n_ngt_elig = (summary['n_valid_nights'] >= N_REC_NIGHTS_MIN).sum()
n_both     = (
    (summary['n_valid_days']   >= N_REC_DAYS_MIN)
    & (summary['n_valid_nights'] >= N_REC_NIGHTS_MIN)
).sum()
n_neither  = (
    (summary['n_valid_days']   < N_REC_DAYS_MIN)
    & (summary['n_valid_nights'] < N_REC_NIGHTS_MIN)
).sum()

print('=== Coverage summary — MDD + BD + HC participants (union window) ===\n')
print(f'Total with any Fitbit data  : {(summary["n_valid_days"] > 0).sum():,} / {n:,}')
print(f'>= {N_REC_DAYS_MIN} valid days   (step-eligible over union window)  : {n_day_elig:,} ({100*n_day_elig/n:.1f}%)')
print(f'>= {N_REC_NIGHTS_MIN} valid nights (sleep-eligible over union window) : {n_ngt_elig:,} ({100*n_ngt_elig/n:.1f}%)')
print(f'Both eligible                                                   : {n_both:,} ({100*n_both/n:.1f}%)')
print(f'Neither eligible                                                : {n_neither:,} ({100*n_neither/n:.1f}%)')
print(f'n_valid_days   -- median={summary["n_valid_days"].median():.0f}  '
      f'mean={summary["n_valid_days"].mean():.1f}  max={summary["n_valid_days"].max()}')
print(f'n_valid_nights -- median={summary["n_valid_nights"].median():.0f}  '
      f'mean={summary["n_valid_nights"].mean():.1f}  max={summary["n_valid_nights"].max()}')

print('\n=== Per-group coverage ===')
for grp in ['MDD', 'BD', 'HC']:
    sub_g = summary[summary['group'] == grp]
    n_g = len(sub_g)
    n_b = (sub_g['n_valid_days'] > 0).sum()
    n_e = (
        (sub_g['n_valid_days']   >= N_REC_DAYS_MIN)
        & (sub_g['n_valid_nights'] >= N_REC_NIGHTS_MIN)
    ).sum()
    print(f'  {grp}: {n_g:,} total | any Fitbit: {n_b:,} | both-eligible: {n_e:,}')

# day_delta sanity check
print('\n=== day_delta range (expected: min ≈ -N_DAYS_WEARABLE) ===')
print(f'valid_days   day_delta: min={vd["day_delta"].min()},  max={vd["day_delta"].max()}')
print(f'valid_nights day_delta: min={vn["day_delta"].min()}, max={vn["day_delta"].max()}')

# Schema
print('\n=== Output schemas ===')
print('valid_days dtypes:')
print(vd.dtypes.to_string())
print('\nvalid_nights dtypes:')
print(vn.dtypes.to_string())
